# VRE Feature Engineering — Linux / memory-aware

This notebook preserves the paper feature definitions while processing one cell at a time. It extracts only CSV files, writes per-cell Parquet partitions, and never constructs the original 78-million-row `masterInput` DataFrame. Feature engineering is CPU-bound; ROCm is used by the model notebook.

In [6]:
from pathlib import Path
import importlib
import pandas as pd

NOTEBOOK_DIR = Path.cwd().resolve()
if NOTEBOOK_DIR.name not in {'ML performance', 'ML-performance'}:
    for folder_name in ('ML-performance', 'ML performance'):
        candidate = NOTEBOOK_DIR / 'VRE_Trustability' / folder_name
        if candidate.is_dir():
            NOTEBOOK_DIR = candidate
            break
WORKSPACE_ROOT = NOTEBOOK_DIR.parents[1]
ZIP_PATH = WORKSPACE_ROOT / 'Open-source-data.zip'
EXTRACT_ROOT = WORKSPACE_ROOT / 'data' / 'raw'
DATA_ROOT = EXTRACT_ROOT / 'Open-source-data'
FEATURE_ROOT = WORKSPACE_ROOT / 'data' / 'processed' / 'features_linux_v1'

import sys
sys.path.insert(0, str(NOTEBOOK_DIR))
from functions import linux_feature_pipeline as pipeline
importlib.reload(pipeline)

print('Notebook:', NOTEBOOK_DIR)
print('Archive:', ZIP_PATH, ZIP_PATH.exists())
print('Raw data:', DATA_ROOT)
print('Features:', FEATURE_ROOT)

Notebook: /home/phil30601/BatteryML/TrustworthyTest/VRE_Trustability/ML-performance
Archive: /home/phil30601/BatteryML/TrustworthyTest/Open-source-data.zip True
Raw data: /home/phil30601/BatteryML/TrustworthyTest/data/raw/Open-source-data
Features: /home/phil30601/BatteryML/TrustworthyTest/data/processed/features_linux_v1


## 1. Extract CSV data

The archive is about 16 GB and expands to about 25 GB including duplicate XLSX files. This step extracts CSV plus the battery index table only. It is idempotent but can take several minutes.

In [7]:
if not (DATA_ROOT / 'Battery index table.txt').exists():
    DATA_ROOT = pipeline.extract_release_csv(ZIP_PATH, EXTRACT_ROOT)
print(DATA_ROOT)

/home/phil30601/BatteryML/TrustworthyTest/data/raw/Open-source-data


## 2. Verify the authoritative cell mapping

The release index explicitly defines 55 cells and the paper test split `[3, 8, 15, 19, 30, 31, 34, 51]`. This resolves the cell 34/38 mismatch in the old notebooks.

In [8]:
cell_index = pipeline.parse_battery_index(DATA_ROOT / 'Battery index table.txt')
display(cell_index)
assert len(cell_index) == 55
print('Paper test cells:', pipeline.PAPER_TEST_CELLS)

,relative_path,protocol
cell_num,,
1,0.33C/Jiayi_GraphiteMTI_LFPXR_C10C3_021623_cha...,0.33
2,0.33C/Jiayi_GraphiteMTI_LFPXR_C10C3_021623_cha...,0.33
3,0.33C/Jiayi_GraphiteMTI_LFPXR_C10C3_022323_cha...,0.33
4,0.33C/Jiayi_GraphiteMTI_LFPXR_C10C3_022723_cha...,0.33
5,0.5C/Jiayi_GraphiteMTI_LFPXR_C10C2_020923_chan...,0.5
6,0.5C/Jiayi_GraphiteMTI_LFPXR_C10C2_020923_chan...,0.5
7,0.5C/Jiayi_GraphiteMTI_LFPXR_C10C2_021123_chan...,0.5
8,0.5C/Jiayi_GraphiteMTI_LFPXR_C10C2_022723_chan...,0.5
9,0.5C/Jiayi_GraphiteMTI_LFPXR_C10C2_092023_chan...,0.5


Paper test cells: [3, 8, 15, 19, 30, 31, 34, 51]


## 3. Build features

For the short GPU pipeline test, build cell 1 (training) and paper test cell 3. After it succeeds, run the model notebook with `FULL_PAPER_SPLIT = False`. For paper-comparable evaluation, set `RUN_FULL = True` here first. With `RESUME = True`, completed partitions are skipped and `manifest.csv` is updated after every cell, so an interrupted run can continue safely. Set `OVERWRITE = True` only when you intentionally want to rebuild selected cells. Full preprocessing is CPU/RAM/I/O bound, not GPU bound.

In [ ]:
RUN_FULL = True
RESUME = True
OVERWRITE = False
SMOKE_CELLS = [1, 3]  # disjoint train/test cells; metrics are not paper-comparable
CELLS = None if RUN_FULL else SMOKE_CELLS

manifest = pipeline.build_dataset(
    data_root=DATA_ROOT,
    output_root=FEATURE_ROOT,
    cells=CELLS,
    resume=RESUME,
    overwrite=OVERWRITE,
)
display(manifest)

Skipping completed cell 01/55
Skipping completed cell 03/55


,cell_num,protocol,raw_rows,feature_rows,cycles,x_path,y_path
0,1,0.33,3226667,816331,99,/home/phil30601/BatteryML/TrustworthyTest/data...,/home/phil30601/BatteryML/TrustworthyTest/data...
1,3,0.33,2692761,814100,99,/home/phil30601/BatteryML/TrustworthyTest/data...,/home/phil30601/BatteryML/TrustworthyTest/data...


## 4. Validate output

Each `x/cell_XXX.parquet` contains `cell_num`, `cycle`, and exactly 65 float32 features. The model notebook reads only the five selected paper features, which sharply lowers peak RAM.

In [10]:
sample_path = FEATURE_ROOT / 'x' / f"cell_{int(manifest.iloc[0].cell_num):03d}.parquet"
sample = pd.read_parquet(sample_path)
feature_columns = [column for column in sample.columns if column not in ('cell_num', 'cycle')]
print(sample_path)
print('shape:', sample.shape)
print('feature count:', len(feature_columns))
print('dtype counts:')
print(sample[feature_columns].dtypes.value_counts())
assert len(feature_columns) == 65
display(sample.head())

/home/phil30601/BatteryML/TrustworthyTest/data/processed/features_linux_v1/x/cell_001.parquet
shape: (816331, 67)
feature count: 65
dtype counts:
float32    65
Name: count, dtype: int64


,cell_num,cycle,Voltage,Current,Throughput,Energy,Capacity,Time,Overpotential,dQ_dV,...,Energy-2,Capacity-2,Time-2,Overpotential-2,dQ_dV-2,dVoltage/dt-2,dEnergy/dt-2,dCapacity/dt-2,dOverpotential/dt-2,Power-2
0,1,2,3.301722,0.00106,11.096774,17.887096,11.096774,1167.140991,0.015730,0.000000,...,17.854839,11.064516,1165.140869,0.015911,0.000000,0.000056,0.016129,0.000000,0.000056,0.0035
1,1,2,3.301717,0.00106,11.096774,17.903225,11.096774,1168.140991,0.015725,0.000000,...,17.870968,11.096774,1166.140869,0.015618,0.000000,0.000023,0.016129,0.032258,-0.000293,0.0035
2,1,2,3.301807,0.00106,11.096774,17.919355,11.096774,1169.140991,0.015815,129.032257,...,17.887096,11.096774,1167.140991,0.015730,0.000000,0.000112,0.016129,0.000000,0.000112,0.0035
3,1,2,3.301815,0.00106,11.129032,17.935484,11.129032,1170.140991,0.015506,193.548386,...,17.903225,11.096774,1168.140991,0.015725,0.000000,-0.000005,0.016129,0.000000,-0.000005,0.0035
4,1,2,3.301893,0.00106,11.129032,17.951612,11.129032,1171.141235,0.015584,193.548386,...,17.919355,11.096774,1169.140991,0.015815,129.032257,0.000090,0.016129,0.000000,0.000090,0.0035
